<img src='./img/up_bann.png' style="display: block; margin: 0 auto;">

*These algorithms come from an **ESA R&D project** and are validated* 

# APEx Data Cube Template
## @ 12th CASSINI  Hackathon

Welcome to the APEx documentation! 📝

The APEx project curates for you the results of various [ESA](https://www.esa.int/) EO projects in a **catalogue of earth observation services** available at the [APEx Algorithm Catalogue](https://algorithm-catalogue.apex.esa.int/). These services are built and maintained by us, which guarantees they have gone through an extensive review and validation process.

The algorithms provided cover a wide spectrum of EO topics with the support of [Sentinel-1](https://sentinels.copernicus.eu/copernicus/sentinel-1), [Sentinel-2](https://sentinels.copernicus.eu/copernicus/sentinel-2) and [Sentinel-3](https://sentinels.copernicus.eu/copernicus/sentinel-3) imagery such as:

- **agriculture** 🌾
- **deforestation** 🪵
- **floods** 💦 
- **change detection services** 🔎
- **wildfires** 🔥 ... etc

This notebook will serve as a template for you to run any algorithm within the Catalogue.

### **Prerequisites:**

#### 🛠️ Dependencies 
As we will be working with the APEx algorithms, it is important to install the following dependencies in your desired environment, wether you want to create it on your local environment or in the [CDSE JupyterLab](https://jupyterhub.dataspace.copernicus.eu/hub/login?next=%2Fhub%2F)

```text
- git
- git+https://github.com/ESA-APEx/apex_algorithms.git # The up-to-date version of apex_algorithms
- openeo
- geopandas
- xarray
```

#### 🛰️ openEO 

As the algorithms run in openEO, it is good to have an understanding of [openEO UDPs](https://open-eo.github.io/openeo-python-client/udp.html). You can find examples that explain how to use the `openeo` library in their [Python Client Examples](https://github.com/Open-EO/openeo-community-examples/tree/main/python).

#### 🔐 Github Authentication 
The `esa_apex_toolbox` library reads the catalogue from the remote GitHub repository, not from local changes.
You can find more informaiton on how to get your token [here](https://esa-apex.github.io/apex_algorithms/toolbox.html#github-authentication).

If your environment defines `GITHUB_TOKEN`, the token cell below picks it up. Otherwise you will be asked for one, and you can press Enter to continue without it.

With the packages installed, we can start. 🚀 🛰️

### Import the required libraries

In [ ]:
from getpass import getpass
import os
from esa_apex_toolbox.algorithms import GithubAlgorithmRepository

import openeo
from IPython.display import display
from shapely.geometry import box
import geopandas as gpd

## 1️⃣ Authentication

We will need to define our github credentials and set the token to access the APEx Catalog:


In [ ]:
github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None


> ##### Remember
> Always **treat a token as a password**: never print it, put it in a committed file, or include it in notebook outputs. Revoke and replace it immediately if exposed; replace it when it expires. Authentication is applied to GitHub API directory requests, not raw catalogue or process-definition downloads.


The algorithms presented in this repository run in the [CDSE openEO federation](https://dataspace.copernicus.eu/analyse/openeo), and thus requires you to have credentials in the [CDSE](https://dataspace.copernicus.eu/).<br>

If you are still not part of the ecosystem, you can do it [here](https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=account-console&tab_id=4qvKTpPuSyA&client_data=eyJydSI6Imh0dHBzOi8vaWRlbnRpdHkuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUvYXV0aC9yZWFsbXMvQ0RTRS9hY2NvdW50LyIsInJ0IjoiY29kZSIsInJtIjoicXVlcnkiLCJzdCI6ImQ4ZTBkNDNmLWNiNDctNDViYi04NGI2LWMwMThlZTBlYmNiYyJ9).

We can establish our connection with the `openeo.connect` function. Once we run the cell, we will be redirected to an interactive login flow in a browser.

In [ ]:
connection = openeo.connect("openeofed.dataspace.copernicus.eu").authenticate_oidc()

> ##### **Note**
> When the openEO Python client library can find a valid refresh token on its system, this will be a non-interactive operation. This makes it a viable option for non-interactive and machine-to-machine authentication.<br>
<br>
> To explore this automated process you can read further [here](https://documentation.dataspace.copernicus.eu/APIs/openEO/authentication.html#refresh-tokens)


## 2️⃣ Accessing an algorithm

First, we need to access our repository through GitHub:

In [ ]:
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)

To access any algorithm we are interested in, we can use the `get_algorithm` function and the **algorithm** `id` available in the [APEx Catalogue](https://algorithm-catalogue.apex.esa.int/). This will allow us to get the link access to identify our algorithm and run it in the backend we have chosen (the CDSE OpenEO federation).

When we visit any algorithm, we will be able to see the `Process ID` entry.

<img src='./img/bsi_params.png' style="display: block; margin: 0 auto;">
<br><br>

Let's call it directly for the **Bare Soil Index** algorithm.

In [ ]:
bsi_algorithm = repo.get_algorithm('bsi')
bsi_algorithm

Once we have accessed the algorithm, we can call the **process URL**.

APEx offers two processing algorithm backends:

- **openEO** : the algorithms are provided as [User Defined Processes](https://open-eo.github.io/openeo-python-client/udp.html) (UDP's) from openEO; and are accessed through the `udp_link` attribute within the algorithm.
- **OGC API**: the algorithms are executed via standard HTTP REST calls and provide STAC item URLs as input.
<br><br>

> ##### Stay tuned!
> New tutorials on how to use both approaches are coming!.

With this information, you will be able to choose the correct backend authentication method you are interested in.

In the case of `bsi`, we will retrieve the `udp_link`.<br>
This attribute will help us feed directly to the CDSE openEO federation all the steps that the backend needs to follow to execute the algorithm smoothly!

By calling the `udp_link.href` attribute for the selected algorithm, we will obtain our link.

In [ ]:
bsi_link = bsi_algorithm.udp_link.href
bsi_link

> ##### Whats an UDP?
> The openEO API enables to store subchains on the back-end as a so called **user-defined process**. They allow to build our own library of reusable building blocks.
> APEx algorithms that work with the openEO backend make use of this UDPs as they standardise parameter names and types, interoperability for the algorithms allowing them to run smoothly across different cloud environments.


## 3️⃣ Setting the parameters.

When we want to run a process over an area and a period of interest, we need to define the parameters.

The openEO UDP backend, understands this area in the dictionary format:

<center>

``` text
{"west": value, "south": value, "east": value, "north": value}
```
</center>

and the time period we are interested in as:

<center>

``` text
["YYYY-MM-DD", "YYYY-MM-DD"]
```
</center>

We set can set an Area of Interest (AOI)  following openEO's format:


In [ ]:
geometry = (33.272095,46.671585,33.609924,46.889762)
shape = {"west": geometry[0], "south": geometry[1], "east": geometry[2], "north": geometry[3]}
start_date, end_date = ["2023-08-24", "2023-08-24"]

# Our area of interest
aoi = gpd.GeoSeries(box(*geometry), crs="EPSG:4326")
aoi.explore(color="red")

## 4️⃣ Preparing the `datacube_from_process`

Once we have defined the parameters to run our algorithm, we can generate our datacube.

For this, we call our openEO `datacube_from_process` function. Then we will set the following parameters:

- `"Name of our Process"` : the name our process graph will display
- `namespace`: the `udp_link` we retrieved from the APEx Catalogue
- `temporal_extent`: the temporal extent we defined. In this case we will onlt retrieve images that are present on the same day
- `spatial_extent` : our polygon dictionary we defined

In [ ]:
bsi = connection.datacube_from_process(
    "BSI",                # the name of our process to identify
    namespace=bsi_link,   # the udp link we retrieved
    temporal_extent=[start_date, end_date], # defined temporal extent
    spatial_extent=shape,                   # our area of interest
)

bsi

Different algorithms can have different number of parameters.<br>

For example; lets have a look at the [Sentinel-1 Coherence](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_coherence) algorithm.

<img src='./img/s1_c_params.png' style="display: block; margin: 0 auto;">
<br><br>

We see more parameters than `temporal_extent` and `spatial_extent`. We even encounter **optional** parameters.

That means, we can either run the following code cell:

In [ ]:
sentinel_1_coherence = connection.datacube_from_process(
    "sentinel1_sar_coherence", # the process name
    namespace = s1c_link,      # UDP link retrieved from the APEx Catalogue
    temporal_extent = [start_date, end_date],  # Starting and Ending date of interest
    temporal_baseline = 12,    # Baseline chosen for our Bursts, 12 days
    burst_id = burst_id_s1,    # Id corresponding for matching Burst
    coherence_window_az = 2,   # the averaging window in azimuth
    coherence_window_rg = 10,  # the averaging window in range
    polarization = pol,        # chosen polarisation
    sub_swath = sub_swath_identifier_s1, # Identifier
)

or, the following option:

In [ ]:
sentinel_1_coherence = connection.datacube_from_process(
    "sentinel1_sar_coherence", # the process name
    namespace = s1c_link,      # UDP link retrieved from the APEx Catalogue
    temporal_extent = [start_date, end_date],  # Starting and Ending date of interest
    temporal_baseline = 12,    # Baseline chosen for our Bursts, 12 days
    coherence_window_az = 2,   # the averaging window in azimuth
    coherence_window_rg = 10,  # the averaging window in range
    polarization = pol,        # chosen polarisation
    spatial_extent = shape,    # our area of interest instead burst_id and sub_swath
)

Both options will work, but will deliver different results.<br>

Always read through the parameters section from every algorithm to understan which parameters will give you the result you are aiming for.

## 5️⃣ Template

This means, the general structure to run a job for the APEx algorithms is:

```text
name_of_the_job = connection.datacube_from_process(
    "Process Name or direct ID",               
    namespace       = process_udp_link,      
    temporal_extent = [start_date, end_date],
    spatial_extent  = shape,
    additional_parameter_1 = value_1, 
    additional_parameter_2 = value_2

)
```

## Takeaways



In this notebook we explored the general structure of the [**APEx Catalogue**](https://algorithm-catalogue.apex.esa.int/), jobs that run in the [CDSE openEO federation](https://documentation.dataspace.copernicus.eu/APIs/openEO/federation/openeo_federation.html) for a specific area of interest.


## Ready for the next application?

[APEx](https://apex.esa.int/) gives you a catalogue of ESA-validated Earth observation algorithms process directly Sentinel data.<br>

Choose an algorithm, define your area and dates, and the cloud platform returns a result you can map and analyse. @ the [12th CASSINI Hackathon](https://www.cassini.eu/hackathons/), you decide what to recognise and what to act on.

<img src='./img/low_bann.png' style="display: block; margin: 0 auto;">